<a href="https://colab.research.google.com/github/kimnayoon12/26-2-DIP-nayoonkim/blob/main/notebook_2026/Lab02_LinearFiltering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
imgpath = '/content/drive/MyDrive/class/DIP1/DIP4E'
#os.listdir(imgpath)


## Histogram Processing

In [4]:
f = plt.imread(imgpath+'/pollen-lowcontrast.tif')
print(f.shape,f.dtype)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/class/DIP1/DIP4E/pollen-lowcontrast.tif'

- histogram equalization

In [ ]:
plt.imshow(f, cmap = 'gray', vmin=0, vmax=255)
plt.axis('off')
plt.colorbar()
plt.show()

In [ ]:
plt.hist(f.flatten())
plt.show()

In [ ]:
def histogram_equalization(image):
    histogram, bins = np.histogram(image.flatten(), bins=256, range=[0,256])
    cdf = histogram.cumsum()
    cdf_min = cdf.min()
    cdf_max = cdf.max()
    cdf_normalized = (cdf - cdf_min) * 255 / (cdf_max - cdf_min)
    cdf_normalized = cdf_normalized.astype(np.uint8)
    equalized_image = cdf_normalized[image]
    return equalized_image

In [ ]:
f_eq = histogram_equalization(f)
plt.hist(f_eq.flatten())
plt.show()

- local histogram equalization

In [ ]:
f = plt.imread(imgpath+'/hidden-symbols.tif')
print(f.shape,f.dtype)

In [ ]:
plt.imshow(f,vmin=0,vmax=255,cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
f_eq = histogram_equalization(f)
plt.imshow(f_eq,vmin=0,vmax=255,cmap='gray')
plt.axis('off')
plt.show()

### Exercise 1.
- 아래의 예는 local histogram equalization을 수행한다.
- 적절한 neighborhood size N을 찾아서 숨겨진 기호들을 모두 확인해보자.
- plt.imshow의 vim, vmax를 수정하여서도 숨겨진 기호들을 확인할 수 있으나, 본 실습에서는 vmax=255로 고정하여 histogram processing을 통해 숨겨진 기호들을 확인하자.

In [ ]:
N = 128
f_eq_local = np.zeros_like(f)
for x in np.arange(0,256,N):
  for y in np.arange(0,256,N):
    f_eq_local[x:x+N,y:y+N] = histogram_equalization(f[x:x+N,y:y+N])

plt.imshow(f_eq_local,vmin=0,vmax=255,cmap='gray')
plt.axis('off')
plt.show()


# Linear Spatial Filtering
## 2D convolution

In [ ]:
import scipy.signal as signal
from skimage import io

In [ ]:
f = io.imread('https://press.hufs.ac.kr/data/file/gift/6f5a7159a604fe7de30498fd76c8741d_JIWi6Gj9_d6e43320ba8ee4ac452deadd5246a2d6db29e853.jpg',as_gray=True)[::2,::2]
print(f.shape,f.dtype)

In [ ]:
plt.imshow(f,cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
w = np.random.randn(3,3)
#w = np.array([[-1,0,1],
#              [-2,0,2],
#              [-1,0,1]])
print(w)

In [ ]:
sz0,sz1 = f.shape
y = np.zeros_like(f)
for i in np.arange(1,sz0-1):
  for j in np.arange(1,sz1-1):
    y[i,j] = np.sum(f[i-1:i+2,j-1:j+2]*w[::-1,::-1])

plt.figure(dpi=150)
plt.subplot(1,2,1)
plt.imshow(f,cmap='gray')
plt.axis('off')
plt.title('original')
plt.subplot(1,2,2)
plt.imshow(y,cmap='gray')
plt.axis('off')
plt.title('filtered')
plt.show()

In [ ]:
y = signal.convolve2d(f,w,mode='same')
y.shape
plt.figure(dpi=150)
plt.subplot(121)
plt.imshow(f,cmap='gray')
plt.axis('off')
plt.title('original')
plt.subplot(122)
plt.imshow(y,cmap='gray')
plt.axis('off')
plt.title('filtered')
plt.show()

## low pass filtering - box kernel

In [ ]:
w = np.ones((3,3))
w = w/np.sum(w)
print(w)

In [ ]:
yb = signal.convolve2d(f,w,mode='same')


In [ ]:
plt.figure(figsize=(16,9))
plt.subplot(1,3,1)
plt.imshow(f,cmap='gray')
plt.axis('off')
plt.title('original')
plt.subplot(1,3,2)
plt.imshow(yb,cmap='gray')
plt.axis('off')
plt.title('filtered')
plt.subplot(1,3,3)
plt.imshow(yb-f,cmap='gray',vmin=-.01,vmax=.01)
plt.axis('off')
plt.title('difference')
plt.show()

In [ ]:
for N in np.array([3,7,15,30]):
  wb = np.ones((N,N))
  wb = wb/np.sum(wb)
  y_b = signal.convolve2d(f,wb,mode='same')

  plt.figure(figsize=(16,9))
  plt.subplot(1,3,1)
  plt.imshow(f,cmap='gray')
  plt.axis('off')
  plt.title('original')
  plt.subplot(1,3,2)
  plt.imshow(y_b,cmap='gray')
  plt.axis('off')
  plt.title(f'filtered - box size: {N}')
  plt.subplot(1,3,3)
  plt.imshow(y_b-f,cmap='gray',vmin=-.01,vmax=.01)
  plt.axis('off')
  plt.title('difference')
  plt.show()

## lowpass filtering - Gaussian kernel

In [ ]:
N = 3
std = 2
s = np.arange(-N,N+1,1)
t = np.arange(-N,N+1,1)
S,T = np.meshgrid(s,t)
wg = np.exp(-(S**2+T**2)/(2*std*std))
wg = wg/np.sum(wg)

plt.figure(figsize=(7,7))
plt.imshow(wg,vmin=0,vmax=0.1,cmap='gray')
plt.axis('off')
plt.colorbar()
plt.show()

In [ ]:
y_g = signal.convolve2d(f,wg,mode='same')

plt.figure(figsize=(16,9))
plt.subplot(1,3,1)
plt.imshow(f,cmap='gray')
plt.axis('off')
plt.title('original')
plt.subplot(1,3,2)
plt.imshow(y_g,cmap='gray')
plt.axis('off')
plt.title('filtered')
plt.subplot(1,3,3)
plt.imshow(y_g-f,cmap='gray',vmin=-.01,vmax=.01)
plt.axis('off')
plt.title('difference')
plt.show()

In [ ]:
s = np.arange(-13,14,1)
t = np.arange(-13,14,1)
S,T = np.meshgrid(s,t)

for std in np.array([1,3,5,7]):
  wg = np.exp(-(S**2+T**2)/(2*std*std))
  wg = wg/np.sum(wg)

  y_g = signal.convolve2d(f,wg,mode='same')

  plt.figure(figsize=(16,9))
  plt.subplot(1,3,1)
  plt.imshow(f,cmap='gray')
  plt.axis('off')
  plt.title('original')
  plt.subplot(1,3,2)
  plt.imshow(y_g,cmap='gray')
  plt.axis('off')
  plt.title(f'filtered - sigma:{std}')
  plt.subplot(1,3,3)
  plt.imshow(y_g-f,cmap='gray',vmin=-.01, vmax=.01)
  plt.axis('off')
  plt.title('difference')
  plt.show()

### Exercise 2.
- 직접 찍은 사진을 grayscale image로 불러와서, kernel size를 3x3, 7x7, 15x15, 30x30 으로 늘려가며 box kernel을 정의하여 LPF를 진행한 결과를 출력하고 kernel size에 따른 연산시간의 차이를 확인하자.

In [ ]:
f = io.imread('https://www.hufs.ac.kr/sites/hufs/images/sub/img08.png',as_gray=True)
print(f.shape,f.dtype)

In [ ]:
import time

In [ ]:
for N in np.array([3,7,15,30]):
  wb = np.ones((N,N))
  wb = wb/np.sum(wb)
  s0 = time.time()
  y_b = signal.convolve2d(f,wb,mode='same')
  compu_time=time.time() - s0
  print('computation time = ', compu_time,' sec')
  plt.figure(figsize=(16,9))
  plt.subplot(1,3,1)
  plt.imshow(f,cmap='gray')
  plt.axis('off')
  plt.title('original')
  plt.subplot(1,3,2)
  plt.imshow(y_b,cmap='gray')
  plt.axis('off')
  plt.title(f'filtered - box size: {N}')
  plt.subplot(1,3,3)
  plt.imshow(y_b-f,cmap='gray',vmin=-.01,vmax=.01)
  plt.axis('off')
  plt.title('difference')
  plt.show()

# highpass filtering



In [ ]:
f = plt.imread(os.path.join(imgpath,'moon-blurred.tif'))
print(f.shape,f.dtype)
plt.figure(figsize=(10,10))
plt.imshow(f,cmap='gray')
plt.axis('off')
plt.title('image')
plt.colorbar()
plt.show()

# Laplacian Kernel (2nd order diff.)


In [ ]:
w1 = np.array([[0,1,0],[1,-4.,1],[0,1,0]])
w2 = np.array([[0,-1,0],[-1,4.,-1],[0,-1,0]])

w3 = np.array([[1,1,1],[1,-8.,1],[1,1,1]])
w4 = np.array([[-1,-1,-1],[-1,8.,-1],[-1,-1,-1]])

print(w1)
print(w2)
print(w3)
print(w4)

In [ ]:
y = signal.convolve2d(f,w2,mode='same')
y.shape
plt.figure(figsize=(16,8))
plt.subplot(121)
plt.imshow(f,cmap='gray')
plt.colorbar()
plt.axis('off')
plt.title('original')
plt.subplot(122)
plt.imshow(y,cmap='gray',vmin=-16,vmax=16)
plt.colorbar()
plt.axis('off')
plt.title('filtered')
plt.show()

### Exercise 3.
- highpass filtering의 결과를 활용하여 blurred image를 sharp하게 개선시켜 보자.
- 자신만의 blurred image를 가져와서, Laplacian kernel을 통한 highpass filtering을 진행하고 이를 이용하여 원본영상의 sharpness를 증대시켜보자.
- w 및 c를 적절하게 변화시켜 가면서 결과를 확인하자.

In [ ]:
f = plt.imread(os.path.join(imgpath,'moon-blurred.tif'))
#f = io.imread('https://superresolution.co/wp-content/uploads/2024/08/superresolution-types-of-blurred-images-num5.webp',as_gray=True)[::2,::2]

w = np.array([[0,-1,0],[-1,4.,-1],[0,-1,0]])
y = signal.convolve2d(f,w,mode='same')
c = 0.01234
g = f + c*(y)
plt.figure(figsize=(18,6))
plt.subplot(131)
plt.imshow(f,cmap='gray')
plt.colorbar()
plt.axis('off')
plt.title('original')
plt.subplot(132)
plt.imshow(g,cmap='gray')
plt.colorbar()
plt.axis('off')
plt.title('sharpening')
plt.subplot(133)
plt.imshow(y,cmap='RdBu_r',vmin=-0.5*np.max(np.abs(y)),vmax=0.5*np.max(np.abs(y)))
plt.colorbar()
plt.axis('off')
plt.title('filtered')
plt.show()

## Sobel (1st order diff.)

In [ ]:
sobel1 = np.array([[-1,-2,-1],[0,0,0],[1,2,1]])
sobel2 = np.array([[-1,0,1],[-2,0,2],[-1,0,1]])
print(sobel1)
print(sobel2)

In [ ]:
g1 = signal.convolve2d(f,sobel1,mode='same')
g2 = signal.convolve2d(f,sobel2,mode='same')
G = np.sqrt(g1**2+g2**2)

plt.figure(figsize=(18,6))
plt.subplot(141)
plt.imshow(f,cmap='gray')
plt.colorbar()
plt.axis('off')
plt.title('original')
plt.subplot(142)
plt.imshow(g1,cmap='RdBu_r',vmin=-128,vmax=128)
plt.colorbar()
plt.axis('off')
plt.title('Gy')
plt.subplot(143)
plt.imshow(g2,cmap='RdBu_r',vmin=-128,vmax=128)
plt.colorbar()
plt.axis('off')
plt.title('Gx')
plt.subplot(144)
plt.imshow(G,cmap='gray',vmin=0,vmax=255)
plt.colorbar()
plt.axis('off')
plt.title('magnitude of gradient')
plt.show()

In [ ]:
f = plt.imread(os.path.join(imgpath,'testpattern512.tif'))

g1 = signal.convolve2d(f,sobel1,mode='same')
g2 = signal.convolve2d(f,sobel2,mode='same')
G = np.sqrt(g1**2+g2**2)

plt.figure(figsize=(18,6))
plt.subplot(131)
plt.imshow(f,cmap='gray')
plt.colorbar()
plt.axis('off')
plt.title('original')
plt.subplot(132)
plt.imshow(g1,cmap='RdBu_r',vmin=-128,vmax=128)
plt.colorbar()
plt.axis('off')
plt.title('Gy')
plt.subplot(133)
plt.imshow(g2,cmap='RdBu_r',vmin=-128,vmax=128)
plt.colorbar()
plt.axis('off')
plt.title('Gx')
plt.show()


In [ ]:
G_ = np.abs(g1) + np.abs(g2)
plt.figure(figsize=(18,6))

plt.subplot(131)
plt.imshow(f,cmap='gray')
plt.colorbar()
plt.axis('off')
plt.title('original')

plt.subplot(132)
plt.imshow(G,cmap='gray',vmin=0,vmax=255)
plt.colorbar()
plt.axis('off')
plt.title('|| G ||')

plt.subplot(133)
plt.imshow(G_,cmap='gray',vmin=0,vmax=255)
plt.colorbar()
plt.axis('off')
plt.title('~ || G ||')

plt.show()
